In [53]:
!uv pip freeze | grep scikit-learn

Using Python 3.12.3 environment at: /home/daniel/github/mlops-zoomcamp-epam/.venv
scikit-learn==1.7.0


In [54]:
!python -V

Python 3.12.3


In [55]:
import pickle
import pandas as pd
from sklearn.metrics import root_mean_squared_error

In [56]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

/home/daniel/github/mlops-zoomcamp-epam/.venv/lib/python3.12/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/daniel/github/mlops-zoomcamp-epam/.venv/lib/python3.12/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [57]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')

    return df

In [58]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [59]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [60]:
print(f"{y_pred.std():.2f}")

6.25


In [61]:
#import argparse
#parser = argparse.ArgumentParser()
#parser.add_argument('--year', type=int, default=2023)
#parser.add_argument('--month', type=int, default=3)
#args = parser.parse_args()

year = 2023
month = 3

In [ ]:
output_file = f'yellow_tripdata_{year:04d}-{month:02d}.parquet'

df_result = pd.DataFrame()
df_result['prediction'] = y_pred
df_result['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False,
)


In [ ]:
df_result.head(10)

In [63]:
from pathlib import Path
print(
    f"{Path(output_file).stat().st_size / 1024 / 1024:.02f} MB"
    )

65.46 MB
